In [27]:
# 라이브러리 & 기본 옵션

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
from scipy import stats
import matplotlib as mpl
import matplotlib.font_manager as fm
import seaborn as sns

# 1) 윈도우 기본 맑은고딕 파일 경로
font_path = r"C:\Windows\Fonts\malgun.ttf"

# 2) 폰트를 matplotlib에 "강제 등록"
fm.fontManager.addfont(font_path)

# 3) 등록한 폰트의 정확한 이름 가져오기
font_name = fm.FontProperties(fname=font_path).get_name()

# 4) matplotlib + seaborn 둘 다에 폰트 강제 적용 (seaborn이 덮는 문제 방지)
mpl.rcParams["font.family"] = font_name
mpl.rcParams["font.sans-serif"] = [font_name]
mpl.rcParams["axes.unicode_minus"] = False

sns.set_theme(style="whitegrid", rc={
    "font.family": font_name,
    "font.sans-serif": [font_name],
    "axes.unicode_minus": False
})

In [28]:
# 0) 데이터 로드
df = pd.read_csv("../../data/Crawling_data/Instagram_Crawling_raw_data.csv")

# 데이터 확인
df.head(1)

,id,author,caption,posted_at,url,images,likes_count,comments_count,sentiment,sentiment_score,keywords,topics,summary,analysis_created_at
0,b05d1511-fa1c-46ec-b654-d4db4cc7db2c,heejour,어느덧 렉스트림이 끝난 지 한 달이 지났네요.\n\n수백 명의 선수들이 자신의 한계...,2026-03-05T12:50:42.000Z,https://www.instagram.com/p/DVgN2enj0oF/,['https://scontent-sjc3-1.cdninstagram.com/v/t...,-1,4,positive,0.92,"['렉스트림', '한계 도전', '응원', '커뮤니티', '감동', '운영 모니터링...","['도전', '커뮤니티', '응원', '운영', '개선', '성취감']","렉스트림 대회가 남긴 도전과 응원, 감동을 회상하며 운영 개선을 약속하고 다음 대회...",2026-03-07T07:53:28.341Z


In [29]:
# 요약 정보
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4739 entries, 0 to 4738
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   4739 non-null   str    
 1   author               4739 non-null   str    
 2   caption              4739 non-null   str    
 3   posted_at            4739 non-null   str    
 4   url                  4739 non-null   str    
 5   images               4739 non-null   str    
 6   likes_count          4739 non-null   int64  
 7   comments_count       4739 non-null   int64  
 8   sentiment            4739 non-null   str    
 9   sentiment_score      4739 non-null   float64
 10  keywords             4739 non-null   str    
 11  topics               4739 non-null   str    
 12  summary              4739 non-null   str    
 13  analysis_created_at  4739 non-null   str    
dtypes: float64(1), int64(2), str(11)
memory usage: 518.5 KB


In [30]:
# 1) posted_at datetime 변환
df["posted_at"] = pd.to_datetime(df["posted_at"], errors="coerce", utc=True)

df.head(1)

,id,author,caption,posted_at,url,images,likes_count,comments_count,sentiment,sentiment_score,keywords,topics,summary,analysis_created_at
0,b05d1511-fa1c-46ec-b654-d4db4cc7db2c,heejour,어느덧 렉스트림이 끝난 지 한 달이 지났네요.\n\n수백 명의 선수들이 자신의 한계...,2026-03-05 12:50:42+00:00,https://www.instagram.com/p/DVgN2enj0oF/,['https://scontent-sjc3-1.cdninstagram.com/v/t...,-1,4,positive,0.92,"['렉스트림', '한계 도전', '응원', '커뮤니티', '감동', '운영 모니터링...","['도전', '커뮤니티', '응원', '운영', '개선', '성취감']","렉스트림 대회가 남긴 도전과 응원, 감동을 회상하며 운영 개선을 약속하고 다음 대회...",2026-03-07T07:53:28.341Z


In [31]:
# 2) KST로 변환
df["posted_at_kst"] = df["posted_at"].dt.tz_convert("Asia/Seoul")

df.head(1)

,id,author,caption,posted_at,url,images,likes_count,comments_count,sentiment,sentiment_score,keywords,topics,summary,analysis_created_at,posted_at_kst
0,b05d1511-fa1c-46ec-b654-d4db4cc7db2c,heejour,어느덧 렉스트림이 끝난 지 한 달이 지났네요.\n\n수백 명의 선수들이 자신의 한계...,2026-03-05 12:50:42+00:00,https://www.instagram.com/p/DVgN2enj0oF/,['https://scontent-sjc3-1.cdninstagram.com/v/t...,-1,4,positive,0.92,"['렉스트림', '한계 도전', '응원', '커뮤니티', '감동', '운영 모니터링...","['도전', '커뮤니티', '응원', '운영', '개선', '성취감']","렉스트림 대회가 남긴 도전과 응원, 감동을 회상하며 운영 개선을 약속하고 다음 대회...",2026-03-07T07:53:28.341Z,2026-03-05 21:50:42+09:00


In [32]:
# 3) 기간 필터도 KST 기준으로
start_date = pd.Timestamp("2025-12-15 00:00:00", tz="Asia/Seoul")
end_date = pd.Timestamp("2026-03-06 23:59:59", tz="Asia/Seoul")

df = df[(df["posted_at_kst"] >= start_date) & (df["posted_at_kst"] <= end_date)].copy()

df.head(1)

,id,author,caption,posted_at,url,images,likes_count,comments_count,sentiment,sentiment_score,keywords,topics,summary,analysis_created_at,posted_at_kst
0,b05d1511-fa1c-46ec-b654-d4db4cc7db2c,heejour,어느덧 렉스트림이 끝난 지 한 달이 지났네요.\n\n수백 명의 선수들이 자신의 한계...,2026-03-05 12:50:42+00:00,https://www.instagram.com/p/DVgN2enj0oF/,['https://scontent-sjc3-1.cdninstagram.com/v/t...,-1,4,positive,0.92,"['렉스트림', '한계 도전', '응원', '커뮤니티', '감동', '운영 모니터링...","['도전', '커뮤니티', '응원', '운영', '개선', '성취감']","렉스트림 대회가 남긴 도전과 응원, 감동을 회상하며 운영 개선을 약속하고 다음 대회...",2026-03-07T07:53:28.341Z,2026-03-05 21:50:42+09:00


In [33]:
# 4) target_date 생성도 KST 기준
df["target_date"] = df["posted_at_kst"].dt.date.astype(str)

df.head(1)

,id,author,caption,posted_at,url,images,likes_count,comments_count,sentiment,sentiment_score,keywords,topics,summary,analysis_created_at,posted_at_kst,target_date
0,b05d1511-fa1c-46ec-b654-d4db4cc7db2c,heejour,어느덧 렉스트림이 끝난 지 한 달이 지났네요.\n\n수백 명의 선수들이 자신의 한계...,2026-03-05 12:50:42+00:00,https://www.instagram.com/p/DVgN2enj0oF/,['https://scontent-sjc3-1.cdninstagram.com/v/t...,-1,4,positive,0.92,"['렉스트림', '한계 도전', '응원', '커뮤니티', '감동', '운영 모니터링...","['도전', '커뮤니티', '응원', '운영', '개선', '성취감']","렉스트림 대회가 남긴 도전과 응원, 감동을 회상하며 운영 개선을 약속하고 다음 대회...",2026-03-07T07:53:28.341Z,2026-03-05 21:50:42+09:00,2026-03-05


In [34]:
# 5) 키워드 포함 여부
df["caption_filled"] = df["caption"].fillna("")
df["caption_lower"] = df["caption_filled"].str.lower()

df["has_kor"] = df["caption_filled"].str.contains("렉스트림", regex=False)
df["has_eng"] = df["caption_lower"].str.contains("rextreme", regex=False)
df["has_any"] = df["has_kor"] | df["has_eng"]

df.head(1)

,id,author,caption,posted_at,url,images,likes_count,comments_count,sentiment,sentiment_score,...,topics,summary,analysis_created_at,posted_at_kst,target_date,caption_filled,caption_lower,has_kor,has_eng,has_any
0,b05d1511-fa1c-46ec-b654-d4db4cc7db2c,heejour,어느덧 렉스트림이 끝난 지 한 달이 지났네요.\n\n수백 명의 선수들이 자신의 한계...,2026-03-05 12:50:42+00:00,https://www.instagram.com/p/DVgN2enj0oF/,['https://scontent-sjc3-1.cdninstagram.com/v/t...,-1,4,positive,0.92,...,"['도전', '커뮤니티', '응원', '운영', '개선', '성취감']","렉스트림 대회가 남긴 도전과 응원, 감동을 회상하며 운영 개선을 약속하고 다음 대회...",2026-03-07T07:53:28.341Z,2026-03-05 21:50:42+09:00,2026-03-05,어느덧 렉스트림이 끝난 지 한 달이 지났네요.\n\n수백 명의 선수들이 자신의 한계...,어느덧 렉스트림이 끝난 지 한 달이 지났네요.\n\n수백 명의 선수들이 자신의 한계...,True,True,True


In [35]:
# 6) 게시물 고유키
# id 우선, 없으면 url
df["unique_post_key"] = df["id"].fillna(df["url"])

df.head(1)

,id,author,caption,posted_at,url,images,likes_count,comments_count,sentiment,sentiment_score,...,summary,analysis_created_at,posted_at_kst,target_date,caption_filled,caption_lower,has_kor,has_eng,has_any,unique_post_key
0,b05d1511-fa1c-46ec-b654-d4db4cc7db2c,heejour,어느덧 렉스트림이 끝난 지 한 달이 지났네요.\n\n수백 명의 선수들이 자신의 한계...,2026-03-05 12:50:42+00:00,https://www.instagram.com/p/DVgN2enj0oF/,['https://scontent-sjc3-1.cdninstagram.com/v/t...,-1,4,positive,0.92,...,"렉스트림 대회가 남긴 도전과 응원, 감동을 회상하며 운영 개선을 약속하고 다음 대회...",2026-03-07T07:53:28.341Z,2026-03-05 21:50:42+09:00,2026-03-05,어느덧 렉스트림이 끝난 지 한 달이 지났네요.\n\n수백 명의 선수들이 자신의 한계...,어느덧 렉스트림이 끝난 지 한 달이 지났네요.\n\n수백 명의 선수들이 자신의 한계...,True,True,True,b05d1511-fa1c-46ec-b654-d4db4cc7db2c


In [36]:
# 7) 날짜별 집계
kor_counts = (
    df[df["has_kor"]]
    .groupby("target_date")["unique_post_key"]
    .nunique()
    .reset_index(name="count")
)
kor_counts["platform"] = "instagram"
kor_counts["keyword"] = "렉스트림"

eng_counts = (
    df[df["has_eng"]]
    .groupby("target_date")["unique_post_key"]
    .nunique()
    .reset_index(name="count")
)
eng_counts["platform"] = "instagram"
eng_counts["keyword"] = "REXTREME"

combined_counts = (
    df[df["has_any"]]
    .groupby("target_date")["unique_post_key"]
    .nunique()
    .reset_index(name="count")
)
combined_counts["platform"] = "instagram"
combined_counts["keyword"] = "combined_unique"

In [37]:
kor_counts.head(5)

,target_date,count,platform,keyword
0,2025-12-15,6,instagram,렉스트림
1,2025-12-16,1,instagram,렉스트림
2,2025-12-17,2,instagram,렉스트림
3,2025-12-18,2,instagram,렉스트림
4,2025-12-21,2,instagram,렉스트림


In [38]:
eng_counts.head(5)

,target_date,count,platform,keyword
0,2025-12-15,5,instagram,REXTREME
1,2025-12-16,1,instagram,REXTREME
2,2025-12-17,2,instagram,REXTREME
3,2025-12-18,1,instagram,REXTREME
4,2025-12-21,3,instagram,REXTREME


In [39]:
combined_counts.head(5)

,target_date,count,platform,keyword
0,2025-12-15,6,instagram,combined_unique
1,2025-12-16,2,instagram,combined_unique
2,2025-12-17,2,instagram,combined_unique
3,2025-12-18,2,instagram,combined_unique
4,2025-12-21,3,instagram,combined_unique


In [40]:
# 8) 합치기
insta_final = pd.concat([kor_counts, eng_counts, combined_counts], ignore_index=True)

display(insta_final)

,target_date,count,platform,keyword
0,2025-12-15,6,instagram,렉스트림
1,2025-12-16,1,instagram,렉스트림
2,2025-12-17,2,instagram,렉스트림
3,2025-12-18,2,instagram,렉스트림
4,2025-12-21,2,instagram,렉스트림
...,...,...,...,...
145,2026-02-19,1,instagram,combined_unique
146,2026-02-20,2,instagram,combined_unique
147,2026-02-25,1,instagram,combined_unique
148,2026-03-01,1,instagram,combined_unique


In [41]:
# 9) 공통 컬럼 추가
insta_final["collected_at"] = "2026-03-23 16:37:00"
insta_final["status"] = "success"
insta_final["note"] = ""

display(insta_final)

,target_date,count,platform,keyword,collected_at,status,note
0,2025-12-15,6,instagram,렉스트림,2026-03-23 16:37:00,success,
1,2025-12-16,1,instagram,렉스트림,2026-03-23 16:37:00,success,
2,2025-12-17,2,instagram,렉스트림,2026-03-23 16:37:00,success,
3,2025-12-18,2,instagram,렉스트림,2026-03-23 16:37:00,success,
4,2025-12-21,2,instagram,렉스트림,2026-03-23 16:37:00,success,
...,...,...,...,...,...,...,...
145,2026-02-19,1,instagram,combined_unique,2026-03-23 16:37:00,success,
146,2026-02-20,2,instagram,combined_unique,2026-03-23 16:37:00,success,
147,2026-02-25,1,instagram,combined_unique,2026-03-23 16:37:00,success,
148,2026-03-01,1,instagram,combined_unique,2026-03-23 16:37:00,success,


In [42]:
# 10) 컬럼 순서 정리
insta_final = insta_final[
    ["target_date", "platform", "keyword", "count", "collected_at", "status", "note"]
].sort_values(["target_date", "platform", "keyword"]).reset_index(drop=True)

insta_final.head(10)

,target_date,platform,keyword,count,collected_at,status,note
0,2025-12-15,instagram,REXTREME,5,2026-03-23 16:37:00,success,
1,2025-12-15,instagram,combined_unique,6,2026-03-23 16:37:00,success,
2,2025-12-15,instagram,렉스트림,6,2026-03-23 16:37:00,success,
3,2025-12-16,instagram,REXTREME,1,2026-03-23 16:37:00,success,
4,2025-12-16,instagram,combined_unique,2,2026-03-23 16:37:00,success,
5,2025-12-16,instagram,렉스트림,1,2026-03-23 16:37:00,success,
6,2025-12-17,instagram,REXTREME,2,2026-03-23 16:37:00,success,
7,2025-12-17,instagram,combined_unique,2,2026-03-23 16:37:00,success,
8,2025-12-17,instagram,렉스트림,2,2026-03-23 16:37:00,success,
9,2025-12-18,instagram,REXTREME,1,2026-03-23 16:37:00,success,


In [43]:
# 11) 전체 날짜 × 키워드 틀 만들기
all_dates = pd.date_range("2025-12-15", "2026-03-06", freq="D").strftime("%Y-%m-%d")

keyword_order = ["렉스트림", "REXTREME", "combined_unique"]

full_grid = pd.MultiIndex.from_product(
    [all_dates, keyword_order],
    names=["target_date", "keyword"]
).to_frame(index=False)

full_grid["platform"] = "instagram"
full_grid["collected_at"] = "2026-03-23 16:37:00"
full_grid["status"] = "success"
full_grid["note"] = ""

In [44]:
all_dates

Index(['2025-12-15', '2025-12-16', '2025-12-17', '2025-12-18', '2025-12-19',
       '2025-12-20', '2025-12-21', '2025-12-22', '2025-12-23', '2025-12-24',
       '2025-12-25', '2025-12-26', '2025-12-27', '2025-12-28', '2025-12-29',
       '2025-12-30', '2025-12-31', '2026-01-01', '2026-01-02', '2026-01-03',
       '2026-01-04', '2026-01-05', '2026-01-06', '2026-01-07', '2026-01-08',
       '2026-01-09', '2026-01-10', '2026-01-11', '2026-01-12', '2026-01-13',
       '2026-01-14', '2026-01-15', '2026-01-16', '2026-01-17', '2026-01-18',
       '2026-01-19', '2026-01-20', '2026-01-21', '2026-01-22', '2026-01-23',
       '2026-01-24', '2026-01-25', '2026-01-26', '2026-01-27', '2026-01-28',
       '2026-01-29', '2026-01-30', '2026-01-31', '2026-02-01', '2026-02-02',
       '2026-02-03', '2026-02-04', '2026-02-05', '2026-02-06', '2026-02-07',
       '2026-02-08', '2026-02-09', '2026-02-10', '2026-02-11', '2026-02-12',
       '2026-02-13', '2026-02-14', '2026-02-15', '2026-02-16', '2026-02-17',

In [45]:
full_grid

,target_date,keyword,platform,collected_at,status,note
0,2025-12-15,렉스트림,instagram,2026-03-23 16:37:00,success,
1,2025-12-15,REXTREME,instagram,2026-03-23 16:37:00,success,
2,2025-12-15,combined_unique,instagram,2026-03-23 16:37:00,success,
3,2025-12-16,렉스트림,instagram,2026-03-23 16:37:00,success,
4,2025-12-16,REXTREME,instagram,2026-03-23 16:37:00,success,
...,...,...,...,...,...,...
241,2026-03-05,REXTREME,instagram,2026-03-23 16:37:00,success,
242,2026-03-05,combined_unique,instagram,2026-03-23 16:37:00,success,
243,2026-03-06,렉스트림,instagram,2026-03-23 16:37:00,success,
244,2026-03-06,REXTREME,instagram,2026-03-23 16:37:00,success,


In [48]:
# 12) 기존 집계 결과와 merge
insta_final_full = full_grid.merge(
    insta_final[["target_date", "platform", "keyword", "count"]],
    on=["target_date", "platform", "keyword"],
    how="left"
)

insta_final_full

,target_date,keyword,platform,collected_at,status,note,count
0,2025-12-15,렉스트림,instagram,2026-03-23 16:37:00,success,,6.0
1,2025-12-15,REXTREME,instagram,2026-03-23 16:37:00,success,,5.0
2,2025-12-15,combined_unique,instagram,2026-03-23 16:37:00,success,,6.0
3,2025-12-16,렉스트림,instagram,2026-03-23 16:37:00,success,,1.0
4,2025-12-16,REXTREME,instagram,2026-03-23 16:37:00,success,,1.0
...,...,...,...,...,...,...,...
241,2026-03-05,REXTREME,instagram,2026-03-23 16:37:00,success,,1.0
242,2026-03-05,combined_unique,instagram,2026-03-23 16:37:00,success,,1.0
243,2026-03-06,렉스트림,instagram,2026-03-23 16:37:00,success,,NaN
244,2026-03-06,REXTREME,instagram,2026-03-23 16:37:00,success,,NaN


In [50]:
# 13) 없는 날짜/키워드는 0으로 채우기
insta_final_full["count"] = insta_final_full["count"].fillna(0).astype(int)

insta_final_full

,target_date,keyword,platform,collected_at,status,note,count
0,2025-12-15,렉스트림,instagram,2026-03-23 16:37:00,success,,6
1,2025-12-15,REXTREME,instagram,2026-03-23 16:37:00,success,,5
2,2025-12-15,combined_unique,instagram,2026-03-23 16:37:00,success,,6
3,2025-12-16,렉스트림,instagram,2026-03-23 16:37:00,success,,1
4,2025-12-16,REXTREME,instagram,2026-03-23 16:37:00,success,,1
...,...,...,...,...,...,...,...
241,2026-03-05,REXTREME,instagram,2026-03-23 16:37:00,success,,1
242,2026-03-05,combined_unique,instagram,2026-03-23 16:37:00,success,,1
243,2026-03-06,렉스트림,instagram,2026-03-23 16:37:00,success,,0
244,2026-03-06,REXTREME,instagram,2026-03-23 16:37:00,success,,0


In [52]:
# 14) 키워드 정렬 순서 강제
insta_final_full["keyword"] = pd.Categorical(
    insta_final_full["keyword"],
    categories=["렉스트림", "REXTREME", "combined_unique"],
    ordered=True
)

insta_final_full = insta_final_full[
    ["target_date", "platform", "keyword", "count", "collected_at", "status", "note"]
].sort_values(["target_date", "platform", "keyword"]).reset_index(drop=True)

insta_final_full.head(20)

,target_date,platform,keyword,count,collected_at,status,note
0,2025-12-15,instagram,렉스트림,6,2026-03-23 16:37:00,success,
1,2025-12-15,instagram,REXTREME,5,2026-03-23 16:37:00,success,
2,2025-12-15,instagram,combined_unique,6,2026-03-23 16:37:00,success,
3,2025-12-16,instagram,렉스트림,1,2026-03-23 16:37:00,success,
4,2025-12-16,instagram,REXTREME,1,2026-03-23 16:37:00,success,
5,2025-12-16,instagram,combined_unique,2,2026-03-23 16:37:00,success,
6,2025-12-17,instagram,렉스트림,2,2026-03-23 16:37:00,success,
7,2025-12-17,instagram,REXTREME,2,2026-03-23 16:37:00,success,
8,2025-12-17,instagram,combined_unique,2,2026-03-23 16:37:00,success,
9,2025-12-18,instagram,렉스트림,2,2026-03-23 16:37:00,success,


In [ ]:
# 15) 저장
insta_final_full.to_csv(
    "../../data/Crawling_data/instagram_daily_counts(25.12.15~26.03.06).csv",
    index=False,
    encoding="utf-8-sig"
)